In [1]:
# !pip install readabs pandas matplotlib
import readabs as ra
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [2]:
try:
    from paths import RAW_DIR
except ImportError:
    RAW_DIR = Path("data/raw")

RAW_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
CAT_NO = "6202.0" #Unemplynent Cat

#Series Map of employmrnt rate
"""
series_map = {
    "A84423134K": "unemployment_rate_trend",
    "A84423050A": "unemployment_rate_seasonally_adjusted",
    "A84423092X": "unemployment_rate_original"
}
"""
START_YEAR = "2000"

RAW_DIR = Path('Data/raw')

In [4]:
def download_unemployment_data3(cat_no, series_id, label):
    #############################################################
    # Input : Cat No, Series_id, and Label
    # Output : Return : Tuple(df Monthly, df quarterly), File on data/raw/ Directory
    ##############################################################

    
    print(f"\nProcessing {label}...")
    
    # Fetch data tuple with readabs
    df, meta = ra.read_abs_series(cat=cat_no, series_id=series_id)
    
    # Ensure it is a DataFrame and filter starting years
    if isinstance(df, pd.Series):
        df = df.to_frame(name='value')
    
    df_filtered = df[df.index >= START_YEAR].copy()
    
    # Process Monthly Data
    # Rename Index to 'date_monthly' and Column to the label
    df_monthly_final = df_filtered.rename_axis('date_monthly').rename(columns={df.columns[0]: label})
    
    monthly_path = RAW_DIR / f"abs_{label}_monthly.csv"
    df_monthly_final.to_csv(monthly_path)
    print(f"Saved: {monthly_path}")

    # Process Quarterly Data
    # Convert using readabs utility
    df_qtly = ra.monthly_to_qtly(df_filtered)
    
    # Handle Series to DataFrame conversion if needed
    if isinstance(df_qtly, pd.Series):
        df_qtly = df_qtly.to_frame()
        
    # Rename Index to 'date_quarterly' and Column to the label
    df_qtly_final = df_qtly.rename_axis('date_quarterly').rename(columns={df_qtly.columns[0]: label})
    
    qtly_path = RAW_DIR / f"abs_{label}_quarterly.csv"
    df_qtly_final.to_csv(qtly_path)
    print(f"Saved: {qtly_path}")

    return df_monthly_final, df_qtly_final

In [5]:
Season_adj = download_unemployment_data3(CAT_NO, 'A84423050A', 'unemployment_rate_seasonally_adjusted')


print("Monthly Employment Rate")
Season_adj[0].head()



Processing unemployment_rate_seasonally_adjusted...
Saved: Data\raw\abs_unemployment_rate_seasonally_adjusted_monthly.csv
Saved: Data\raw\abs_unemployment_rate_seasonally_adjusted_quarterly.csv
Monthly Employment Rate


,unemployment_rate_seasonally_adjusted
date_monthly,
2000-01,6.768208
2000-02,6.619256
2000-03,6.565315
2000-04,6.376558
2000-05,6.414548


In [6]:
print("Quarterly Employment Rate")
Season_adj[1].head()

Quarterly Employment Rate


,unemployment_rate_seasonally_adjusted
date_quarterly,
2000Q1,6.650926
2000Q2,6.302043
2000Q3,5.989691
2000Q4,6.172038
2001Q1,6.367179
